<a href="https://colab.research.google.com/github/piyushanangude/Codetech/blob/main/Task_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [62]:
# =======================
# FULL PROJECT (ONE CELL)
# =======================

# Install dependencies (for Colab)
!pip install flask pyngrok scikit-learn pandas numpy

# =======================
# IMPORT LIBRARIES
# =======================
import pandas as pd
import numpy as np
from flask import Flask, request, jsonify
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
import pickle
from pyngrok import ngrok

# =======================
# LOAD DATASET
# =======================
df = pd.read_csv("HousePriceData (3).csv")

# Clean column names (remove spaces if any)
df.columns = df.columns.str.strip()

# =======================
# PREPROCESSING
# =======================
# Handle missing values
df.fillna(df.mean(numeric_only=True), inplace=True)

# Encode Location (text → number)
le = LabelEncoder()
df['Location'] = le.fit_transform(df['Location'])

# =======================
# FEATURES & TARGET (IMPORTANT)
# =======================
X = df[['FloorArea', 'BedRooms', 'Location']]   # ✅ 3 FEATURES
y = df['Price']

print("Training features:", X.columns)

# =======================
# TRAIN MODEL
# =======================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = RandomForestRegressor()
model.fit(X_train, y_train)

# Save model
pickle.dump(model, open("model.pkl", "wb"))

# =======================
# FLASK APP
# =======================
app = Flask(__name__)

model = pickle.load(open("model.pkl", "rb"))

@app.route('/')
def home():
    return '''
    <h2>House Price Prediction</h2>
    <form action="/predict_form" method="post">
        Floor Area: <input name="FloorArea"><br>
        Bedrooms: <input name="BedRooms"><br>
        Location: <input name="Location"><br>
        <input type="submit" value="Predict">
    </form>
    '''

# =======================
# PREDICT API (MATCHES DATASET)
# =======================
@app.route('/predict_form', methods=['POST'])
def predict_form():
    try:
        floor = float(request.form['FloorArea'])
        beds = int(request.form['BedRooms'])
        location = int(request.form['Location'])

        features = np.array([[floor, beds, location]])
        prediction = model.predict(features)

        return f"<h3>Predicted Price: {prediction[0]}</h3>"

    except Exception as e:
        return f"Error: {str(e)}"

# =======================
# NGROK SETUP (COLAB)
# =======================
ngrok.set_auth_token("3CAh1kInoow6X6WNfYPRCs90wgs_4aMkspJQAhT84MDWPaCgg")  # 🔴 Add your token
public_url = ngrok.connect(5000)
print("🌐 Public URL:", public_url)

# =======================
# RUN APP
# =======================
app.run()

Training features: Index(['FloorArea', 'BedRooms', 'Location'], dtype='object')
🌐 Public URL: NgrokTunnel: "https://facecloth-goofy-hypnotist.ngrok-free.dev" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [10/Apr/2026 16:14:33] "GET / HTTP/1.1" 200 -
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
INFO:werkzeug:127.0.0.1 - - [10/Apr/2026 16:14:45] "POST /predict_form HTTP/1.1" 200 -
